# Inventing Neural Networks

In the Gradient Descent chapter you taught a computer to fit a line: pick
weights, measure the error, nudge every weight downhill, repeat. That one loop
is more powerful than it looks — the only thing standing between you and a
*neural network* is the shape of the function you feed it.

In this chapter you will:

1. Turn a line into a yes/no classifier — and discover what the field calls it.
2. Trip over the most classic training bug and fix it.
3. Chain small functions together and watch weights reshape curves.
4. Take on a problem no straight line can solve — a realistic **income-tax
   rule** — and beat it three ways, ending with the activation function that
   powers modern deep learning.

You need `numpy`, `matplotlib`, and your gradient-descent instincts from
Chapter 15. Nothing else.

## Part 1 — From Numbers to Yes/No

Here are five people's heights, labeled `1` for adult and `0` for teenager:

```python
heights = np.array([171, 173, 163, 162, 160])
labels  = np.array([1, 1, 0, 0, 0])
```

A line `m*x + c` outputs *any* number — 0.37, 12.5, −3. But the answer we want
is yes/no. We need something that squashes a number into a decision.

### Exercise 1.1 — The step function

The bluntest squasher possible: `step_function(z)` returns `1` if `z >= 0`,
else `0`. Write it.

(In 1958 this exact function, wrapped around a weighted sum, was called the
**perceptron** — the first artificial neuron.)

**Write your solution in `exercises/ex_1_1_step_function.py`**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

heights = np.array([171, 173, 163, 162, 160])
labels  = np.array([1, 1, 0, 0, 0])

def step_function(z):
    pass  # replace this


assert step_function(3) == 1
assert step_function(0) == 1
assert step_function(-0.5) == 0
print("step_function: OK")

### Exercise 1.2 — The smooth step

Here's the problem with `step_function`: imagine gradient descent trying to
tune `m` and `c` inside `step_function(m*x + c)`. Nudge `m` a little — the
output usually doesn't change *at all* (it's still exactly 0 or exactly 1).
The gradient is zero almost everywhere. Gradient descent is blind; there is no
downhill to follow.

We need a step with a *slope* — a function that goes from 0 to 1 but smoothly,
so every nudge changes the output a little. The classic choice:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Write `sigmoid(z)` using `np.exp`. Predict before you run: what is
`sigmoid(0)`? What does it approach for very large positive `z`? Very negative?

**Write your solution in `exercises/ex_1_2_sigmoid.py`**

In [ ]:
def sigmoid(z):
    pass  # replace this


assert sigmoid(0) == 0.5
assert sigmoid(100) > 0.999
assert sigmoid(-100) < 0.001
assert abs(sigmoid(2) - 0.8808) < 1e-3

zs = np.linspace(-8, 8, 100)
plt.plot(zs, [step_function(z) for z in zs], label="step")
plt.plot(zs, sigmoid(zs), label="sigmoid")
plt.legend(); plt.grid(alpha=0.3); plt.title("A step you can slide down")
plt.show()

### Exercise 1.3 — An error we can descend

Our model is now `sigmoid(m*x + c)` — a line squashed into (0, 1). Read its
output as "how confident am I that this is an adult."

Write `sigmoid_error(m, c, x, y)` — the mean squared error between
`sigmoid(m*x + c)` and the true labels. This is your familiar MSE with a
sigmoid inside; numpy broadcasting handles the whole array at once.

**Write your solution in `exercises/ex_1_3_sigmoid_error.py`**

In [ ]:
def sigmoid_error(m, c, x, y):
    pass  # replace this


# perfect confidence -> error ~ 0: huge m centered between the groups
assert sigmoid_error(10, -10 * 166.5, heights, labels) < 1e-6
# always-unsure model (m=0, c=0 -> everyone 0.5) -> error 0.25
assert abs(sigmoid_error(0, 0, heights, labels) - 0.25) < 1e-9
print("sigmoid_error: OK")

### Exercise 1.4 — Train it… and watch it refuse to learn

Below is the finite-difference gradient helper you've built twice before,
provided as-is. Write `train(m, c, x, y, learning_rate, epochs)` — the
standard loop: compute both gradients of `sigmoid_error`, step both weights
against them, return the final `(m, c)`.

Then run it on the raw heights, starting from `m=0.12, c=0.3`. The tests
expect something strange: **the error starts at 0.6 and ends at 0.6.** One
hundred epochs, no progress.

Before moving on, diagnose it. Print `sigmoid(0.12 * 171 + 0.3)`. That's
`sigmoid(20.8)` — which is `0.99999999…`. Every prediction is glued to 1.0 on
sigmoid's flat plateau, where the slope is essentially zero. Zero slope means
zero gradient means zero learning. This is called **saturation**, and it is
the single most classic failure mode in neural networks.

**Write your solution in `exercises/ex_1_4_train.py`**

In [ ]:
# Provided — run this cell as-is: finite-difference gradients for (m, c).
def gradients_mc(err_fn, m, c, x, y, h=1e-8):
    base = err_fn(m, c, x, y)
    grad_m = (err_fn(m + h, c, x, y) - base) / h
    grad_c = (err_fn(m, c + h, x, y) - base) / h
    return grad_m, grad_c

In [ ]:
def train(m, c, x, y, learning_rate, epochs):
    pass  # replace this


err_before = sigmoid_error(0.12, 0.3, heights, labels)
m1, c1 = train(0.12, 0.3, heights, labels, learning_rate=0.1, epochs=100)
err_after = sigmoid_error(m1, c1, heights, labels)
print(f"error: {err_before:.4f} -> {err_after:.4f}")
assert err_after > 0.5                        # stuck!

print("the smoking gun:", sigmoid(0.12 * 171 + 0.3))
gm, gc = gradients_mc(sigmoid_error, 0.12, 0.3, heights, labels)
print("gradients:", gm, gc)                   # ~zero: nothing to descend

### Exercise 1.5 — The fix: feed the neuron small numbers

The plateau problem came from `m*x` being huge, because heights are ~170. Keep
the inputs near zero and sigmoid operates on its steep middle section, where
gradients live.

Subtract 165 from the heights (so they become `[6, 8, -2, -3, -5]`), train
again from the same start with `learning_rate=0.5, epochs=2000`, and this time
build the actual classifier: `classify(x_centered)` returns 1 when
`sigmoid(m*x + c) >= 0.5`.

**Write your solution in `exercises/ex_1_5_classify.py`**

In [ ]:
centered = heights - 165

m2, c2 = train(0.12, 0.3, centered, labels, learning_rate=0.5, epochs=2000)
err = sigmoid_error(m2, c2, centered, labels)
print(f"error after centering: {err:.4f}   m={m2:.3f} c={c2:.3f}")
assert err < 0.05

def classify(x_centered):
    pass  # replace this


preds = np.array([classify(x) for x in centered])
print("predictions:", preds, "  labels:", labels)
assert (preds == labels).all()
print("logistic regression: OK — you built a neuron")

**What you just invented** is **logistic regression** — and it is exactly one
**neuron**: weighted sum, plus bias, through a squashing *activation
function*. The centering trick you used is why every deep-learning tutorial
starts by normalizing inputs.

## Part 2 — Wiring Small Functions Together

Your Part 1 function draws one boundary. But many real problems need more than
one boundary — what if the answer depends on combining several simple decisions?

Let's find out what happens when you feed one small function's output into
another.

### Exercise 2.1 — Follow the wires

Implement this four-step graph as `compute_graph(x, w10, w11, w20, w21, w30, w31)`:

```
y1 = step_function(w11 * x + w10)
y2 = sigmoid(y1 * w21 + w20)
y3 = x * y2
y  = w31 * y3 + w30
```

Work the first test by hand before coding: with all weights 1 and `x = 2`,
`y1 = step(3) = 1`, `y2 = sigmoid(2) ≈ 0.8808`, `y3 ≈ 1.7616`, `y ≈ 2.7616`.

**Write your solution in `exercises/ex_2_1_compute_graph.py`**

In [ ]:
def compute_graph(x, w10, w11, w20, w21, w30, w31):
    pass  # replace this


assert abs(compute_graph(2, 1, 1, 1, 1, 1, 1) - 2.7616) < 1e-3
assert abs(compute_graph(-5, 1, 1, 1, 1, 1, 1) - (-2.6555)) < 1e-3
print("compute_graph: OK")

### Exercise 2.2 — Weights reshape the curve

Run the provided cell: it plots `y` versus `x` for the graph above with one
set of weights. Then:

1. Change the weights to a second set of your choosing and plot both curves
   together.
2. Notice the *kink* where the step function switches — the weights `w10, w11`
   decide **where** it happens, the later weights decide **what** happens on
   each side.
3. Invent your **own** graph of a similar size — chain 3–5 operations mixing
   `step_function`, `sigmoid`, `*`, `+` — and plot it for two weight settings.

What you just built — a recipe of chained steps with tunable weights — is
called a **computation graph**. A neural network is nothing more than a large
one. The *graph* fixes what shapes are possible; the *weights* choose one
shape; gradient descent finds the weights.

In [ ]:
# Provided — run this cell as-is, then modify per the exercise.
xs = np.linspace(-10, 10, 400)
ys = [compute_graph(x, 1, 1, 1, 1, 1, 1) for x in xs]
plt.plot(xs, ys, label="w = all ones")

# your second weight set here:
# ys2 = [compute_graph(x, ...) for x in xs]
# plt.plot(xs, ys2, label="w = your choice")

plt.legend(); plt.grid(alpha=0.3)
plt.title("Same graph, different weights, different curve")
plt.show()

## Part 3 — The Tax Problem: When No Line Will Do

A country taxes income like this: **nothing on the first 5000; 30% on
everything above.** So `tax = 0.3 * (income − 5000)` if income > 5000, else 0.
Plotted, it's a hockey stick — flat, then a straight climb, with a sharp kink
at 5000.

The cell below generates 200 people and — remembering Part 1's saturation
lesson! — scales incomes and taxes down to the 0–1 range before we model
anything. Run it.

In [ ]:
# Provided — run this cell as-is.
rng = np.random.default_rng(42)
income = rng.uniform(0, 10000, 200)
tax = np.where(income > 5000, 0.30 * (income - 5000), 0.0)

x = income / 10000     # scaled income, 0..1
y = tax / 1500         # scaled tax, 0..1 (max tax is 1500)

plt.scatter(x, y, s=12, alpha=0.6)
plt.xlabel("income (scaled)"); plt.ylabel("tax (scaled)")
plt.title("The hockey stick no line can fit")
plt.grid(alpha=0.3); plt.show()

### Exercise 3.1 — Prove the line fails

Fit the best straight line to `(x, y)` — you may use `np.polyfit(x, y, 1)`
(you derived what it computes, least squares, in Chapter 14). Compute its RMSE
and look at what it predicts for low incomes.

The tests capture two indictments: the RMSE stays above 0.1 no matter what,
and for incomes below 3000 the line predicts **negative tax** — the government
paying you. The problem isn't the fitting; the *shape* of the model is wrong.

In [ ]:
m_line, c_line = ...   # replace this

line_pred = m_line * x + c_line
line_rmse = np.sqrt(np.mean((line_pred - y) ** 2))
print(f"best line: m={m_line:.3f} c={c_line:.3f} RMSE={line_rmse:.4f}")

low = x < 0.3
print("average prediction for low incomes:", np.mean(line_pred[low]).round(3))
assert line_rmse > 0.1
assert np.mean(line_pred[low]) < 0    # negative tax!

plt.scatter(x, y, s=12, alpha=0.5)
plt.plot(np.sort(x), m_line * np.sort(x) + c_line, "r-", label="best line")
plt.legend(); plt.grid(alpha=0.3); plt.show()

### Exercise 3.2 — Idea 1: first decide, *then* predict

A human would solve this in two steps: *is this person taxable at all?* If
no, the answer is 0. If yes, it's a simple line. Let's build exactly that.

Step one is a yes/no question — a job for the neuron from Part 1.

Create `is_taxable = (tax > 0).astype(int)` and train a sigmoid classifier
`sigmoid(m*x + c)` on `(x, is_taxable)` — your `train` from Exercise 1.4
works unchanged (inputs are already scaled; use `learning_rate=2.0,
epochs=3000`, starting from `m=1, c=0`). Report the accuracy of thresholding
at 0.5, and print where the decision boundary sits: `-c/m` (in scaled income;
multiply by 10000 to see it in currency — it should land near 5000).

In [ ]:
is_taxable = ...   # replace this

mk, ck = ...   # replace this — train the classifier

pred_taxable = (sigmoid(mk * x + ck) >= 0.5).astype(int)
accuracy = np.mean(pred_taxable == is_taxable)
print(f"accuracy: {accuracy:.3f}   boundary at income ~{-ck / mk * 10000:.0f}")
assert accuracy >= 0.97
print("classifier: OK")

### Exercise 3.3 — The pipeline: classify, then regress

Step two: for the taxable people *only*, the relationship is a perfect
straight line. Fit a line on just the rows where `is_taxable == 1` — on the
scaled data it should come out at exactly `m=2, c=−1` (check: at `x=1`, i.e.
income 10000, scaled tax is `2·1 − 1 = 1`, i.e. 1500 ✓).

Then chain the two models into `predict_tax(xi)`:

- if the classifier says taxable → return the line's prediction,
- else → return `0.0`.

Compare its RMSE with the line's from 3.1.

**What you invented:** a *gated* model — one network deciding which expert
handles the input. Scaled up, this idea is called a **mixture of experts**,
and it's inside several frontier LLMs. It's also the honest answer to "how do
I put an `if` inside a machine-learning model."

**Write your solution in `exercises/ex_3_3_predict_tax.py`**

In [ ]:
mt, ct = ...   # replace this — line fit on taxable rows only

assert abs(mt - 2) < 1e-6 and abs(ct + 1) < 1e-6

def predict_tax(xi):
    pass  # replace this


pipe_pred = np.array([predict_tax(xi) for xi in x])
pipe_rmse = np.sqrt(np.mean((pipe_pred - y) ** 2))
print(f"pipeline RMSE: {pipe_rmse:.5f}   (line was {line_rmse:.4f})")
assert pipe_rmse < line_rmse / 5
print("pipeline: OK")

### Exercise 3.4 — Idea 2: one network, end to end

The pipeline needed *us* to know the problem splits into classify-then-regress.
Could a single network figure the shape out by itself? Try the smallest one
imaginable — two neurons in a chain:

$$\hat{y} = w_1 \cdot \sigma(m x + c) + w_0$$

Neuron 1 squashes, neuron 2 scales and shifts. Four weights.

First write `fit_net(err_fn, params, x, y, lr, epochs, h=1e-6)` — your
gradient-descent loop generalized to a *list* of parameters: each epoch,
finite-difference each parameter (copy the list, nudge one entry), then step
all of them. (You wrote the two-parameter and many-weight versions in Chapter
15 — this is the same music.)

Then define `net_error(params, x, y)` for the two-neuron model and fit it
from `[1.0, 0.0, 1.0, 0.0]` with `lr=0.5, epochs=5000`.

The tests expect an honest middle result: clearly better than the line
(RMSE ≈ 0.05 vs 0.14), clearly worse than the pipeline. Look at the plot to
see why — sigmoid rises and then **flattens**, but real tax keeps climbing.
A smooth step times a constant can imitate a kink, not a ramp.

**Write your solution in `exercises/ex_3_4_fit_net.py`**

In [ ]:
def fit_net(err_fn, params, x, y, lr, epochs, h=1e-6):
    pass  # replace this


def net_error(params, x, y):
    pass  # replace this — m, c, w1, w0 = params


params = fit_net(net_error, [1.0, 0.0, 1.0, 0.0], x, y, lr=0.5, epochs=5000)
sig_rmse = np.sqrt(net_error(params, x, y))
print(f"sigmoid net RMSE: {sig_rmse:.4f}")
assert sig_rmse < line_rmse
assert sig_rmse > pipe_rmse

m_, c_, w1_, w0_ = params
xs = np.sort(x)
plt.scatter(x, y, s=12, alpha=0.5)
plt.plot(xs, w1_ * sigmoid(m_ * xs + c_) + w0_, "r-", lw=2, label="sigmoid net")
plt.legend(); plt.grid(alpha=0.3)
plt.title("Better than a line — but it flattens out")
plt.show()

### Exercise 3.5 — The right activation: ReLU

Look at the tax rule one more time: *zero below a threshold, a straight ramp
above it.* There is an activation function that **is** that shape:

$$\text{relu}(z) = \max(z, 0)$$

Write `relu(z)` (use `np.maximum` so it works on arrays), then reuse your
`fit_net` on the model $\hat{y} = w_1 \cdot \text{relu}(m x + c) + w_0$ —
same starting point, same settings, only the activation changed.

The tests expect near-perfection: the network can now represent the true rule
*exactly* (one solution: `m=2, c=−1, w1=1, w0=0` — but gradient descent may
find an equivalent scaling, e.g. `m·w1 = 2`).

**The reveal:** ReLU — the *rectified linear unit* — is the default
activation of modern deep learning, powering nearly every large model.
Sigmoid saturates at both ends (you met that plateau in Exercise 1.4); ReLU
never saturates on the positive side, and stacks of ReLU neurons build
arbitrary piecewise-linear shapes — kinks upon kinks, like the one you just
fit with a single neuron.

**Write your solution in `exercises/ex_3_5_relu.py`**

In [ ]:
def relu(z):
    pass  # replace this


assert relu(3.5) == 3.5
assert relu(-2) == 0
assert (relu(np.array([-1, 0, 2])) == np.array([0, 0, 2])).all()

def relu_error(params, x, y):
    pass  # replace this


params_r = fit_net(relu_error, [1.0, 0.0, 1.0, 0.0], x, y, lr=0.5, epochs=5000)
relu_rmse = np.sqrt(relu_error(params_r, x, y))
print(f"relu net RMSE: {relu_rmse:.5f}")
assert relu_rmse < sig_rmse
assert relu_rmse < 0.02

m_, c_, w1_, w0_ = params_r
xs = np.sort(x)
plt.scatter(x, y, s=12, alpha=0.5)
plt.plot(xs, w1_ * relu(m_ * xs + c_) + w0_, "g-", lw=2, label="relu net")
plt.legend(); plt.grid(alpha=0.3)
plt.title("One ReLU neuron — the exact shape of the law")
plt.show()

print(f"\nfinal standings — line: {line_rmse:.4f}  sigmoid net: {sig_rmse:.4f}"
      f"  relu net: {relu_rmse:.5f}  pipeline: {pipe_rmse:.5f}")

### Exercise 3.6 — Explorations (open-ended)

No tests here — experiment and observe:

1. Replace `sigmoid` with `np.tanh` in the two-neuron net. Better or worse on
   the tax data? (tanh is a sigmoid stretched to output −1…1.)
2. Add a second ReLU neuron: $\hat{y} = w_1\,\text{relu}(m_1 x + c_1) +
   w_2\,\text{relu}(m_2 x + c_2) + w_0$ (seven parameters — `fit_net` doesn't
   care). Invent a tax rule with *two* brackets and see if it can fit that
   too.
3. Feed `x` in again at the output: $\hat{y} = w_1\,\text{relu}(mx+c) +
   w_2 x + w_0$. (Networks with such shortcuts are called *residual* — the
   trick that made very deep networks trainable.)

## Summary — what you invented

| You built | The field calls it |
|-----------|--------------------|
| `step_function(m*x + c)` | the perceptron (1958) |
| `sigmoid(m*x + c)` + gradient descent | logistic regression = one neuron |
| stuck training on raw heights | saturation / vanishing gradients — fixed by input scaling |
| chained small functions with weights | a computation graph |
| classify-then-regress gate | a mixture of experts |
| `fit_net` over a parameter list | training a neural network |
| `relu` | the default activation of deep learning |

Every deep-learning framework is these same pieces industrialized: bigger
graphs, exact chain-rule gradients instead of finite differences (you verified
they agree in Chapter 15), and hardware that multiplies matrices fast. When
you're ready to see the same ideas at production scale, read through
[cloudxlab/GPT-from-scratch](https://github.com/cloudxlab/GPT-from-scratch/tree/master) —
tokenization, embeddings, attention (the `uM.T @ uM` trick from the
Recommender chapter!), and a training loop you now recognize.

Next chapter: what happens when the input isn't one number but a whole
image — **convolutions**.